# Chapter 22 — Tracing with LangSmith

Hands-On LangChain. Uses the **LangSmith** SDK.

Have you ever built an AI agent and wondered what's actually happening under the hood? When an agent
fails or produces unexpected results, debugging can feel like investigating a black box. This is where
**LangSmith tracing** becomes invaluable: it automatically logs each step of your application's workflow
so you can see exactly what ran, with what inputs and outputs, how long it took, and what it cost.

We'll build a small **recipe-discovery agent** — it searches the web for a recipe, then has an LLM turn
the results into a clean cooking guide — and instrument it with tracing.

Companion notebook for Chapter 22. Requires a LangSmith account and `LANGSMITH_API_KEY` in `.env`. Set
`MODEL_PROVIDER` to `ollama` or `groq`; Groq chat requires `GROQ_API_KEY`.

```bash
ollama pull qwen2.5:7b  # only when using Ollama chat
pip install langsmith langchain-ollama langchain-groq ollama groq langchain-community ddgs python-dotenv -q
```

## 22.1 The agent architecture

The agent is a simple two-step pipeline: **search the web** (DuckDuckGo) to gather information about a
recipe, then **pass it to an LLM** that formats everything into a helpful cooking guide. Simple as it is,
watching the data flow through it reveals a lot about performance and cost.

## 22.2 Setting up LangSmith tracing

Configuration order matters: we set the environment variables **before** importing the libraries, or
tracing won't initialize. `LANGSMITH_TRACING="true"` tells LangSmith to collect execution data
automatically, and `LANGSMITH_PROJECT` decides where the traces appear in your dashboard.

We use DuckDuckGo search — no API key required, perfect for a tutorial.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # loads LANGSMITH_API_KEY

# Set tracing config FIRST (before importing the tracing-aware libraries)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "recipe-discovery-agent"

import warnings; warnings.filterwarnings("ignore")
from langchain_community.tools import DuckDuckGoSearchResults
search_tool = DuckDuckGoSearchResults(num_results=3, output_format="list")

## 22.3 Using the shared Ollama chat model

With LangChain's Ollama chat model, LangSmith tracing automatically captures model calls while
`LANGSMITH_TRACING` is enabled. Our `@traceable` decorators add application-level spans, and the model
invocation appears as a child run beneath the function that calls it.

In [ ]:
from model_provider import get_chat_model

llm = get_chat_model(temperature=1)

## 22.4 Building the traceable functions

The magic happens through the **`@traceable`** decorator. Every decorated function automatically logs its
inputs, outputs, execution time, and position in the call hierarchy.

First, the prompt that turns messy search results into a structured guide — it instructs the model to act
as a professional chef and nutritionist.

In [3]:
prompt = """You are a professional chef and nutritionist. Using the search results below, write a
clear, structured cooking guide for the user's request.

Include these sections:
- Overview: a short description of the dish
- Ingredients: the key ingredients and their role
- Difficulty: easy / intermediate / advanced, with a one-line reason
- Cooking time: an approximate total time
- Tips: 2-3 helpful tips

User request: {query}

Search results:
{context}
"""

**Searching for recipes.** We append "recipe" to the query — a simple trick that sharply improves relevance (we want cooking instructions, not encyclopedia articles). Error handling keeps it production-friendly.

In [4]:
from langsmith import traceable

@traceable
def search_recipes(query):
    """Search for recipe information."""
    try:
        results = search_tool.invoke({"query": f"{query} recipe"})
        if isinstance(results, list):
            return "\n".join(str(item) for item in results)
        return str(results)
    except Exception as e:
        return f"Could not find recipe information: {e}"

**Generating the cooking guide.** This is where the agent adds value — turning scattered search results into a focused guide with the shared Ollama chat model. LangSmith captures the model call as part of the trace.

In [ ]:
@traceable
def create_recipe_guide(query, context):
    """Generate a helpful recipe guide."""
    formatted = prompt.format(query=query, context=context)
    messages = [
        ("system", formatted),
        ("human", f"Help me cook: {query}"),
    ]
    return llm.invoke(messages).content

**Orchestrating the pipeline.** The main function ties it together. It becomes the **root span** of the trace, with `search_recipes` and `create_recipe_guide` appearing as child spans beneath it — a natural dependency chain (we must search before we can generate).

In [6]:
@traceable
def recipe_assistant(query):
    """Main function: search recipes and create a guide."""
    context = search_recipes(query)
    guide = create_recipe_guide(query, context)
    return guide

## 22.5 Observing & understanding the tracing results

Let's run the agent. Every step is logged to your LangSmith project automatically.

In [ ]:
guide = recipe_assistant("Thai green curry")
print(guide[:900])

Open your LangSmith dashboard (the **recipe-discovery-agent** project) and you'll see a complete
trace whose hierarchy matches our code: `recipe_assistant` at the top, with `search_recipes` and the
Ollama chat-model call (inside `create_recipe_guide`) nested beneath.

The screenshots below are from the source author's original hosted-model run. Provider details, timing,
and usage metrics in those figures won't match a local Ollama run.

![The recipe_assistant root span in LangSmith](../images/ch22/Figure%201.png)

*Figure 22.1 — The root span `recipe_assistant` in LangSmith. The run tree on the left mirrors the code: the orchestrator calls `search_recipes`, then `create_recipe_guide` (which contains the chat-model call). The right pane shows the final formatted output.*

The trace shows how each step contributes to the total latency; in the source author's run, search took
about 1 second and model generation took about 6.82 seconds. Your local timings will depend on your
hardware, Ollama model, and search results.

![The search_recipes child span](../images/ch22/Figure%202.png)

*Figure 22.2 — Drilling into the `search_recipes` span: its exact input (the query) and output (the raw DuckDuckGo results that become context for the LLM), and the source run's search latency.*

![The create_recipe_guide child span with token usage and cost](../images/ch22/Figure%203.png)

*Figure 22.3 — The source run's `create_recipe_guide` model span exposes the prompt, response, and usage metrics. Local Ollama runs may report different token metadata and do not incur the hosted-provider charge shown in the original figure.*

**Why this visibility matters.** Without tracing, debugging a failure is guesswork — you might see an
error but not which component caused it. With LangSmith, every failure traces back to its source: did the
search return nothing? Did the LLM time out? Did malformed data crash the parser? Optimization becomes
data-driven, and — most useful for prompt engineering — the **exact prompt sent to the model alongside
its response** is available in the trace, making iteration far faster than trial and error.

In production these traces become even more valuable: you can monitor success rates and latency
across thousands of queries, track cost per user or query type, and tie **user feedback** directly to the
specific trace that produced a bad answer.

With just three `@traceable` decorators, our recipe agent went from an opaque black box to a transparent,
debuggable workflow. In the next chapter we use the **Playground & Prompts Hub** to iterate on prompts
like the one above.